# California Mortgage Lending Analysis — Exploratory Data Analysis

This notebook explores 2024 California HMDA mortgage application data to identify patterns in lending outcomes, borrower characteristics, loan types, and pricing.

The analysis focuses on approval and denial patterns, differences across loan purposes and loan types, borrower financial characteristics, and potential relationships between mortgage pricing and application outcomes.

## 1. Setup and Load Cleaned Data

In [1]:
import pandas as pd

In [2]:
file_path = "../data/processed/hmda_ca_2024_cleaned.parquet"

df = pd.read_parquet(file_path)

In [3]:
df.shape

(1026119, 38)

## 2. Define the Analysis Sample

Before analyzing approval and denial patterns, we first examine the distribution of application outcomes and define the appropriate analytical sample.

In [4]:
df["action_label"].value_counts()

action_label
Loan Originated                       511270
Denied                                180539
Withdrawn                             142155
Purchased Loan                         96647
Closed for Incompleteness              55626
Approved, Not Accepted                 36102
Preapproval Approved, Not Accepted      3196
Preapproval Denied                       584
Name: count, dtype: int64

### 2.1 Define Decisioned Applications

For the main approval and denial analysis, we restrict the dataset to applications that received a standard lending decision:

- `1` — Loan Originated
- `2` — Approved, Not Accepted
- `3` — Denied

Withdrawn applications, incomplete applications, purchased loans, and preapproval outcomes are excluded because they are not directly comparable to standard approval and denial decisions.

In [5]:
decisioned_df = df[df["action_taken"].isin([1, 2, 3])].copy()

decisioned_df.shape

(727911, 38)

In [7]:
decisioned_df.shape

(727911, 38)

### 2.2 Create Approval Outcome

Within the decisioned application sample, applications are classified as approved when the action taken is either:

- `1` — Loan Originated
- `2` — Approved, Not Accepted

Applications with action code `3` are classified as denied.

In [8]:
decisioned_df["approved"] = decisioned_df["action_taken"].isin([1, 2])

In [9]:
decisioned_df["approved"].value_counts()

approved
True     547372
False    180539
Name: count, dtype: int64

### 2.3 Calculate Approval and Denial Rates

Using the decisioned application sample, we calculate the overall approval and denial rates.

In [10]:
approval_rate = decisioned_df["approved"].mean()
denial_rate = 1 - approval_rate

approval_rate, denial_rate

(np.float64(0.7519765465833048), np.float64(0.24802345341669518))

In [11]:
print(f"Approval Rate: {approval_rate:.2%}")
print(f"Denial Rate: {denial_rate:.2%}")

Approval Rate: 75.20%
Denial Rate: 24.80%


### 2.4 Validate the Analysis Sample

We validate that the approval and denial classifications fully reconcile with the decisioned application sample.

In [12]:
approved_count = decisioned_df["approved"].sum()
denied_count = (~decisioned_df["approved"]).sum()

assert approved_count + denied_count == len(decisioned_df)
assert abs(approval_rate + denial_rate - 1) < 1e-10

print(f"Decisioned Applications: {len(decisioned_df):,}")
print(f"Approved: {approved_count:,}")
print(f"Denied: {denied_count:,}")
print("Validation passed.")

Decisioned Applications: 727,911
Approved: 547,372
Denied: 180,539
Validation passed.


## 3. Approval Patterns by Loan Purpose

This section examines whether approval outcomes differ across mortgage loan purposes.

### 3.1 Approval Rate by Loan Purpose

We compare the number of decisioned applications and approval rates across different loan purposes.

In [13]:
decisioned_df["loan_purpose_label"].value_counts()

loan_purpose_label
Home Purchase           310494
Cash-out Refinancing    125021
Other Purpose           107230
Home Improvement         96339
Refinancing              88318
Not Applicable             509
Name: count, dtype: int64

In [14]:
loan_purpose_summary = (
    decisioned_df
    .groupby("loan_purpose_label")
    .agg(
        applications=("approved", "count"),
        approved=("approved", "sum"),
        approval_rate=("approved", "mean")
    )
)

loan_purpose_summary

,applications,approved,approval_rate
loan_purpose_label,,,
Cash-out Refinancing,125021,87185,0.697363
Home Improvement,96339,58235,0.604480
Home Purchase,310494,274579,0.884329
Not Applicable,509,454,0.891945
Other Purpose,107230,59566,0.555498
Refinancing,88318,67353,0.762619


In [15]:
loan_purpose_summary["denied"] = (
    loan_purpose_summary["applications"]
    - loan_purpose_summary["approved"]
)

loan_purpose_summary["approval_rate_pct"] = (
    loan_purpose_summary["approval_rate"] * 100
)

In [16]:
loan_purpose_summary = (
    loan_purpose_summary
    .sort_values("approval_rate_pct", ascending=False)
)

loan_purpose_summary

,applications,approved,approval_rate,denied,approval_rate_pct
loan_purpose_label,,,,,
Not Applicable,509,454,0.891945,55,89.194499
Home Purchase,310494,274579,0.884329,35915,88.432949
Refinancing,88318,67353,0.762619,20965,76.261917
Cash-out Refinancing,125021,87185,0.697363,37836,69.736284
Home Improvement,96339,58235,0.604480,38104,60.448001
Other Purpose,107230,59566,0.555498,47664,55.549753


### 3.2 Validate the Loan Purpose Summary

We verify that the loan purpose groups fully reconcile with the decisioned application sample before interpreting differences in approval rates.

In [17]:
assert loan_purpose_summary["applications"].sum() == len(decisioned_df)
assert loan_purpose_summary["approved"].sum() == approved_count
assert loan_purpose_summary["denied"].sum() == denied_count

print(f"Applications: {loan_purpose_summary['applications'].sum():,}")
print(f"Approved: {loan_purpose_summary['approved'].sum():,}")
print(f"Denied: {loan_purpose_summary['denied'].sum():,}")
print("Validation passed.")

Applications: 727,911
Approved: 547,372
Denied: 180,539
Validation passed.


### 3.3 Key Findings by Loan Purpose

Among the major loan purpose categories, home purchase applications had the highest approval rate at approximately 88.4%, while other-purpose applications had the lowest approval rate at approximately 55.5%.

Refinancing applications had a moderately high approval rate of about 76.3%, followed by cash-out refinancing at 69.7% and home improvement loans at 60.4%.

The `Not Applicable` category had the highest observed approval rate overall, but it contained only 509 applications, so it should be interpreted cautiously due to its much smaller sample size.

## 4. Approval Patterns by Loan Type

This section examines whether approval outcomes differ across mortgage loan types.

### 4.1 Approval Rate by Loan Type

We compare the number of decisioned applications and approval rates across conventional, FHA, VA, and USDA/RHS loans.

In [18]:
decisioned_df["loan_type_label"].value_counts()

loan_type_label
Conventional    617108
FHA              78540
VA               31840
USDA/RHS           423
Name: count, dtype: int64

In [19]:
loan_type_summary = (
    decisioned_df
    .groupby("loan_type_label")
    .agg(
        applications=("approved", "count"),
        approved=("approved", "sum"),
        approval_rate=("approved", "mean")
    )
)

loan_type_summary

,applications,approved,approval_rate
loan_type_label,,,
Conventional,617108,458685,0.743282
FHA,78540,61978,0.789127
USDA/RHS,423,342,0.808511
VA,31840,26367,0.828109


In [20]:
loan_type_summary["denied"] = (
    loan_type_summary["applications"]
    - loan_type_summary["approved"]
)

loan_type_summary["approval_rate_pct"] = (
    loan_type_summary["approval_rate"] * 100
)

loan_type_summary = (
    loan_type_summary
    .sort_values("approval_rate_pct", ascending=False)
)

loan_type_summary

,applications,approved,approval_rate,denied,approval_rate_pct
loan_type_label,,,,,
VA,31840,26367,0.828109,5473,82.810930
USDA/RHS,423,342,0.808511,81,80.851064
FHA,78540,61978,0.789127,16562,78.912656
Conventional,617108,458685,0.743282,158423,74.328156


### 4.2 Validate the Loan Type Summary

We verify that the loan type groups fully reconcile with the decisioned application sample before interpreting differences in approval rates.

In [21]:
assert loan_type_summary["applications"].sum() == len(decisioned_df)
assert loan_type_summary["approved"].sum() == approved_count
assert loan_type_summary["denied"].sum() == denied_count

print(f"Applications: {loan_type_summary['applications'].sum():,}")
print(f"Approved: {loan_type_summary['approved'].sum():,}")
print(f"Denied: {loan_type_summary['denied'].sum():,}")
print("Validation passed.")

Applications: 727,911
Approved: 547,372
Denied: 180,539
Validation passed.


### 4.3 Key Findings by Loan Type

Among the major loan types, VA applications had the highest observed approval rate at approximately 82.8%, followed by FHA loans at 78.9% and conventional loans at 74.3%.

USDA/RHS loans had an observed approval rate of approximately 80.9%, but the category contained only 423 decisioned applications, so the result should be interpreted cautiously due to the small sample size.

These differences are descriptive and should not be interpreted as evidence that loan type itself causes higher or lower approval rates, because borrower characteristics and eligibility requirements may differ across loan types.

## 5. Borrower Financial Characteristics

In [22]:
decisioned_df["dti_category"].value_counts(dropna=False)

dti_category
36%-<50%    355596
30%-<36%     76686
50%-60%      73196
NaN          71929
20%-<30%     62927
>60%         59055
<20%         28522
Name: count, dtype: int64

### 5.1 Approval Rate by Debt-to-Income Ratio

We compare approval rates across debt-to-income ratio categories to examine how lending outcomes vary with borrower leverage.

In [23]:
dti_summary = (
    decisioned_df
    .dropna(subset=["dti_category"])
    .groupby("dti_category")
    .agg(
        applications=("approved", "count"),
        approved=("approved", "sum"),
        approval_rate=("approved", "mean")
    )
)

dti_summary

,applications,approved,approval_rate
dti_category,,,
20%-<30%,62927,52698,0.837447
30%-<36%,76686,66418,0.866103
36%-<50%,355596,308493,0.867538
50%-60%,73196,44777,0.611741
<20%,28522,19736,0.691957
>60%,59055,4769,0.080755


In [24]:
dti_order = [
    "<20%",
    "20%-<30%",
    "30%-<36%",
    "36%-<50%",
    "50%-60%",
    ">60%"
]

dti_summary = dti_summary.reindex(dti_order)

dti_summary["denied"] = (
    dti_summary["applications"]
    - dti_summary["approved"]
)

dti_summary["approval_rate_pct"] = (
    dti_summary["approval_rate"] * 100
)

dti_summary

,applications,approved,approval_rate,denied,approval_rate_pct
dti_category,,,,,
<20%,28522,19736,0.691957,8786,69.195709
20%-<30%,62927,52698,0.837447,10229,83.744657
30%-<36%,76686,66418,0.866103,10268,86.610333
36%-<50%,355596,308493,0.867538,47103,86.753788
50%-60%,73196,44777,0.611741,28419,61.174108
>60%,59055,4769,0.080755,54286,8.075523


### 5.2 Validate DTI Coverage

We verify that the categorized DTI observations and missing DTI values together reconcile with the full decisioned application sample.

In [25]:
dti_missing_count = decisioned_df["dti_category"].isna().sum()

assert dti_summary["applications"].sum() + dti_missing_count == len(decisioned_df)

print(f"Applications with DTI category: {dti_summary['applications'].sum():,}")
print(f"Applications with missing DTI: {dti_missing_count:,}")
print(f"Total decisioned applications: {len(decisioned_df):,}")
print("Validation passed.")

Applications with DTI category: 655,982
Applications with missing DTI: 71,929
Total decisioned applications: 727,911
Validation passed.


### 5.3 Key Findings by Debt-to-Income Ratio

Approval rates varied substantially across DTI categories.

Applications with DTI ratios between 30% and 50% had the highest observed approval rates, at approximately 86.6%–86.8%.

Approval rates declined sharply for higher-DTI applicants, falling to approximately 61.2% for DTI ratios between 50% and 60%, and only 8.1% for applicants with DTI ratios above 60%.

The relationship was not strictly linear, however. Applications with DTI below 20% had an approval rate of approximately 69.2%, which was lower than several moderate-DTI groups.

These patterns are descriptive and may reflect differences in borrower profiles, loan products, underwriting criteria, or other characteristics not controlled for in this analysis.

### 5.4 Income and Approval Outcomes

We examine how borrower income differs between approved and denied applications. Because the HMDA income variable contains extreme values, we first review its distribution before selecting appropriate summary statistics.

In [26]:
decisioned_df["income"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    6.817740e+05
mean     3.306819e+02
std      3.913688e+04
min     -1.801000e+04
1%       0.000000e+00
5%       3.700000e+01
25%      9.600000e+01
50%      1.500000e+02
75%      2.380000e+02
95%      5.550000e+02
99%      1.347000e+03
max      2.555358e+07
Name: income, dtype: float64

In [27]:
income_by_outcome = (
    decisioned_df
    .groupby("approved")["income"]
    .agg(["count", "median"])
)

income_by_outcome

,count,median
approved,,
False,172967,114.0
True,508807,161.0


### 5.5 Approval Rate by Income Group

To examine the relationship between borrower income and approval outcomes more clearly, we group reported income into broad income ranges and compare approval rates across those groups.

In [28]:
income_bins = [
    float("-inf"),
    50,
    100,
    150,
    250,
    500,
    1000,
    float("inf")
]

income_labels = [
    "<$50k",
    "$50k-$100k",
    "$100k-$150k",
    "$150k-$250k",
    "$250k-$500k",
    "$500k-$1M",
    ">$1M"
]

decisioned_df["income_group"] = pd.cut(
    decisioned_df["income"],
    bins=income_bins,
    labels=income_labels,
    right=False
)

In [29]:
decisioned_df["income_group"].value_counts(dropna=False)

income_group
$150k-$250k    184074
$100k-$150k    159113
$50k-$100k     131290
$250k-$500k    115204
<$50k           49769
NaN             46137
$500k-$1M       31008
>$1M            11316
Name: count, dtype: int64

In [30]:
income_summary = (
    decisioned_df
    .dropna(subset=["income_group"])
    .groupby("income_group", observed=True)
    .agg(
        applications=("approved", "count"),
        approved=("approved", "sum"),
        approval_rate=("approved", "mean")
    )
)

income_summary["denied"] = (
    income_summary["applications"]
    - income_summary["approved"]
)

income_summary["approval_rate_pct"] = (
    income_summary["approval_rate"] * 100
)

income_summary

,applications,approved,approval_rate,denied,approval_rate_pct
income_group,,,,,
<$50k,49769,23640,0.474994,26129,47.499447
$50k-$100k,131290,83813,0.638381,47477,63.838068
$100k-$150k,159113,120979,0.760334,38134,76.033385
$150k-$250k,184074,148213,0.805182,35861,80.518161
$250k-$500k,115204,96678,0.839190,18526,83.918961
$500k-$1M,31008,26240,0.846233,4768,84.623323
>$1M,11316,9244,0.816896,2072,81.689643


### 5.6 Validate Income Coverage

We verify that applications with categorized income and missing income together reconcile with the full decisioned application sample.

In [31]:
income_missing_count = decisioned_df["income_group"].isna().sum()

assert income_summary["applications"].sum() + income_missing_count == len(decisioned_df)

print(f"Applications with income group: {income_summary['applications'].sum():,}")
print(f"Applications with missing income: {income_missing_count:,}")
print(f"Total decisioned applications: {len(decisioned_df):,}")
print("Validation passed.")

Applications with income group: 681,774
Applications with missing income: 46,137
Total decisioned applications: 727,911
Validation passed.


### 5.7 Key Findings by Income

Approved applications had a higher median reported income than denied applications, at approximately $161,000 compared with $114,000.

Approval rates generally increased across the income groups, rising from approximately 47.5% for applicants reporting less than $50,000 in income to approximately 84.6% for those reporting between $500,000 and $1 million.

The pattern was not strictly monotonic, as the approval rate declined slightly to approximately 81.7% among applications reporting more than $1 million in income.

These results describe an association between reported income and approval outcomes and should not be interpreted as a causal effect of income alone. Other factors, including debt-to-income ratio, loan amount, loan type, and borrower characteristics, may also contribute to lending decisions.

### 5.8 Loan-to-Value Ratio and Approval Outcomes

We examine how approval outcomes vary across loan-to-value ratios. Because the cleaned dataset contains a small number of extreme LTV values, we first review the distribution and use the existing data-quality flag to separate clearly extreme observations from the main analysis.

In [32]:
decisioned_df["loan_to_value_ratio"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    6.850420e+05
mean     1.609751e+04
std      1.321402e+07
min      6.000000e-03
1%       8.333000e+00
5%       2.241000e+01
25%      5.357110e+01
50%      7.191200e+01
75%      8.301000e+01
95%      9.833000e+01
99%      1.016900e+02
max      1.093680e+10
Name: loan_to_value_ratio, dtype: float64

In [33]:
decisioned_df["ltv_extreme_flag"].value_counts(dropna=False)

ltv_extreme_flag
False    684759
<NA>      42869
True        283
Name: count, dtype: Int64

### 5.9 Define the LTV Analysis Sample

For the main LTV analysis, we include applications with non-missing LTV values that were not flagged as extreme during data cleaning. Extreme observations remain preserved in the cleaned dataset but are excluded from this comparison to prevent a very small number of implausibly large values from distorting the analysis.

In [34]:
ltv_analysis_df = decisioned_df[
    decisioned_df["loan_to_value_ratio"].notna()
    & (decisioned_df["ltv_extreme_flag"] == False)
].copy()

ltv_analysis_df.shape

(684759, 40)

In [37]:
ltv_bins = [
    float("-inf"),
    60,
    80,
    90,
    95,
    100,
    float("inf")
]

ltv_labels = [
    "<60%",
    "60%-<80%",
    "80%-<90%",
    "90%-<95%",
    "95%-<100%",
    "100%+"
]

ltv_analysis_df["ltv_group"] = pd.cut(
    ltv_analysis_df["loan_to_value_ratio"],
    bins=ltv_bins,
    labels=ltv_labels,
    right=False
)

ltv_analysis_df["ltv_group"].value_counts(dropna=False)

ltv_group
60%-<80%     221863
<60%         220425
80%-<90%     113572
95%-<100%     69866
100%+         29720
90%-<95%      29313
Name: count, dtype: int64

### 5.10 Approval Rate by Loan-to-Value Ratio

We compare approval rates across LTV groups using applications with non-missing, non-extreme LTV values.

In [38]:
ltv_summary = (
    ltv_analysis_df
    .groupby("ltv_group", observed=True)
    .agg(
        applications=("approved", "count"),
        approved=("approved", "sum"),
        approval_rate=("approved", "mean")
    )
)

ltv_summary["denied"] = (
    ltv_summary["applications"]
    - ltv_summary["approved"]
)

ltv_summary["approval_rate_pct"] = (
    ltv_summary["approval_rate"] * 100
)

ltv_summary

,applications,approved,approval_rate,denied,approval_rate_pct
ltv_group,,,,,
<60%,220425,156520,0.710083,63905,71.008279
60%-<80%,221863,166660,0.751184,55203,75.118429
80%-<90%,113572,89978,0.792255,23594,79.225513
90%-<95%,29313,23187,0.791014,6126,79.101423
95%-<100%,69866,60044,0.859417,9822,85.941660
100%+,29720,24475,0.823520,5245,82.351952


### 5.11 Validate LTV Coverage

We verify that the main LTV analysis sample, missing LTV observations, and extreme LTV observations together reconcile with the full decisioned application sample.

In [39]:
ltv_missing_count = decisioned_df["loan_to_value_ratio"].isna().sum()

ltv_extreme_count = (
    decisioned_df["ltv_extreme_flag"] == True
).sum()

assert (
    len(ltv_analysis_df)
    + ltv_missing_count
    + ltv_extreme_count
    == len(decisioned_df)
)

assert ltv_summary["applications"].sum() == len(ltv_analysis_df)

print(f"Applications in LTV analysis: {len(ltv_analysis_df):,}")
print(f"Applications with missing LTV: {ltv_missing_count:,}")
print(f"Applications with extreme LTV: {ltv_extreme_count:,}")
print(f"Total decisioned applications: {len(decisioned_df):,}")
print("Validation passed.")

Applications in LTV analysis: 684,759
Applications with missing LTV: 42,869
Applications with extreme LTV: 283
Total decisioned applications: 727,911
Validation passed.


### 5.12 Key Findings by Loan-to-Value Ratio

Approval rates varied across LTV groups but did not follow a simple inverse relationship with leverage.

Approval rates generally increased from approximately 71.0% for applications with LTV below 60% to approximately 85.9% for applications with LTV between 95% and 100%. The approval rate then declined modestly to approximately 82.4% for applications with LTV of 100% or higher.

These results suggest that observed approval outcomes are influenced by more than LTV alone. Differences in loan programs, borrower characteristics, underwriting standards, and other factors may contribute to the pattern.

The main LTV comparison excludes 283 observations previously flagged as extreme during data cleaning, while preserving those records in the cleaned dataset.

## 6. Mortgage Pricing and Interest Rates

This section examines mortgage pricing patterns across lending outcomes and loan characteristics.

### 6.1 Interest Rate Distribution

We first review the distribution of reported interest rates within the decisioned application sample before comparing pricing across groups.

In [40]:
decisioned_df["interest_rate"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    543207.000000
mean          7.386186
std           1.863680
min           0.000000
1%            1.000000
5%            5.375000
25%           6.250000
50%           6.990000
75%           8.250000
95%          10.875000
99%          13.375000
max          22.539000
Name: interest_rate, dtype: float64

In [41]:
decisioned_df["interest_rate"].isna().sum()

np.int64(184704)

### 6.2 Interest Rate Availability by Lending Outcome

Before comparing interest rates between approved and denied applications, we examine whether reported interest-rate availability differs across lending outcomes.

In [42]:
interest_rate_coverage = (
    decisioned_df
    .groupby("approved")
    .agg(
        applications=("approved", "count"),
        interest_rate_available=("interest_rate", "count")
    )
)

interest_rate_coverage["coverage_rate_pct"] = (
    interest_rate_coverage["interest_rate_available"]
    / interest_rate_coverage["applications"]
    * 100
)

interest_rate_coverage

,applications,interest_rate_available,coverage_rate_pct
approved,,,
False,180539,0,0.000000
True,547372,543207,99.239092


### 6.3 Define the Interest Rate Analysis Sample

Reported interest rates are available for 99.2% of approved applications but are entirely missing for denied applications in the decisioned sample.

Because denied applications do not have reported interest rates, a direct comparison of interest rates between approved and denied applications would not be meaningful. The subsequent interest-rate analysis therefore focuses on approved applications with reported interest rates.

In [43]:
interest_rate_df = decisioned_df[
    decisioned_df["approved"]
    & decisioned_df["interest_rate"].notna()
].copy()

interest_rate_df.shape

(543207, 40)

### 6.4 Interest Rate by Loan Type

Among approved applications with reported interest rates, we compare mortgage pricing across loan types using both the median and mean interest rate.

In [44]:
interest_rate_by_loan_type = (
    interest_rate_df
    .groupby("loan_type_label")
    .agg(
        applications=("interest_rate", "count"),
        median_interest_rate=("interest_rate", "median"),
        mean_interest_rate=("interest_rate", "mean")
    )
)

interest_rate_by_loan_type

,applications,median_interest_rate,mean_interest_rate
loan_type_label,,,
Conventional,454599,7.125,7.631525
FHA,61936,6.250,6.213411
USDA/RHS,318,6.000,5.130261
VA,26354,5.990,5.937573


### 6.5 Interest Rate by Loan Purpose

We also compare reported interest rates across loan purposes among approved applications with available pricing information.

In [45]:
interest_rate_by_purpose = (
    interest_rate_df
    .groupby("loan_purpose_label")
    .agg(
        applications=("interest_rate", "count"),
        median_interest_rate=("interest_rate", "median"),
        mean_interest_rate=("interest_rate", "mean")
    )
    .sort_values("median_interest_rate")
)

interest_rate_by_purpose

,applications,median_interest_rate,mean_interest_rate
loan_purpose_label,,,
Refinancing,66720,6.490,6.791171
Home Purchase,272768,6.625,6.654884
Cash-out Refinancing,86610,7.625,7.978635
Not Applicable,453,7.990,8.363530
Home Improvement,57474,8.500,8.740315
Other Purpose,59182,9.000,9.237984


### 6.6 Key Findings on Interest Rates

Interest-rate reporting differed substantially by lending outcome. Reported interest rates were available for approximately 99.2% of approved applications but were unavailable for denied applications. Therefore, the pricing analysis was restricted to approved applications with reported interest rates.

Among the major loan types, conventional loans had the highest median reported interest rate at approximately 7.13%, compared with 6.25% for FHA loans and 5.99% for VA loans. USDA/RHS loans had a median rate of 6.00%, but the category contained only 318 observations with reported pricing and should therefore be interpreted cautiously.

Pricing also varied substantially by loan purpose. Refinancing and home purchase applications had relatively low median interest rates of approximately 6.49% and 6.63%, respectively. Cash-out refinancing had a higher median rate of approximately 7.63%, while home improvement and other-purpose loans had median rates of approximately 8.50% and 9.00%.

These differences are descriptive and may reflect differences in loan products, borrower characteristics, market conditions, and other underwriting or pricing factors.

### 6.7 Rate Spread Distribution

We examine the distribution of rate spread among approved applications and use the existing data-quality flag to identify extreme observations before conducting group comparisons.

In [46]:
interest_rate_df["rate_spread"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    481643.000000
mean          0.724818
std           2.642429
min        -600.000000
1%           -5.170000
5%           -1.059000
25%          -0.152000
50%           0.369000
75%           1.246000
95%           3.890000
99%           6.270000
max         350.000000
Name: rate_spread, dtype: float64

In [47]:
interest_rate_df["rate_spread_extreme_flag"].value_counts(dropna=False)

rate_spread_extreme_flag
False    481596
<NA>      61564
True         47
Name: count, dtype: Int64

### 6.8 Define the Rate Spread Analysis Sample

For the main rate spread analysis, we include approved applications with non-missing rate spread values that were not flagged as extreme during data cleaning. Extreme observations remain preserved in the cleaned dataset but are excluded from this comparison to prevent a small number of unusual values from distorting the analysis.

In [48]:
rate_spread_df = interest_rate_df[
    interest_rate_df["rate_spread"].notna()
    & (interest_rate_df["rate_spread_extreme_flag"] == False)
].copy()

rate_spread_df.shape

(481596, 40)

In [49]:
rate_spread_df["rate_spread"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    481596.000000
mean          0.708969
std           1.667585
min          -8.698000
1%           -5.170000
5%           -1.059000
25%          -0.152000
50%           0.369000
75%           1.245000
95%           3.890000
99%           6.260000
max          48.706000
Name: rate_spread, dtype: float64

### 6.9 Rate Spread by Loan Type

We compare rate spreads across loan types among approved applications with non-missing, non-extreme rate spread values.

In [50]:
rate_spread_by_loan_type = (
    rate_spread_df
    .groupby("loan_type_label")
    .agg(
        applications=("rate_spread", "count"),
        median_rate_spread=("rate_spread", "median"),
        mean_rate_spread=("rate_spread", "mean")
    )
)

rate_spread_by_loan_type

,applications,median_rate_spread,mean_rate_spread
loan_type_label,,,
Conventional,398960,0.461,0.843621
FHA,56561,0.314,0.324738
USDA/RHS,309,0.046,-0.964019
VA,25766,-0.542,-0.512461


### 6.10 Validate Rate Spread Coverage

We verify that the rate spread analysis sample, missing rate spread observations, and extreme rate spread observations reconcile with the full interest-rate analysis sample.

In [51]:
rate_spread_missing_count = interest_rate_df["rate_spread"].isna().sum()

rate_spread_extreme_count = (
    interest_rate_df["rate_spread_extreme_flag"] == True
).sum()

assert (
    len(rate_spread_df)
    + rate_spread_missing_count
    + rate_spread_extreme_count
    == len(interest_rate_df)
)

assert (
    rate_spread_by_loan_type["applications"].sum()
    == len(rate_spread_df)
)

print(f"Applications in rate spread analysis: {len(rate_spread_df):,}")
print(f"Applications with missing rate spread: {rate_spread_missing_count:,}")
print(f"Applications with extreme rate spread: {rate_spread_extreme_count:,}")
print(f"Total interest-rate analysis applications: {len(interest_rate_df):,}")
print("Validation passed.")

Applications in rate spread analysis: 481,596
Applications with missing rate spread: 61,564
Applications with extreme rate spread: 47
Total interest-rate analysis applications: 543,207
Validation passed.


### 6.11 Key Findings on Rate Spread

Rate spreads also varied across loan types. Conventional loans had the highest median rate spread among the major loan types at approximately 0.46 percentage points, followed by FHA loans at approximately 0.31 percentage points.

VA loans had a negative median rate spread of approximately -0.54 percentage points, distinguishing them from the other major loan types in the analysis.

USDA/RHS loans had a median rate spread near zero, but only 309 observations were available for this comparison, so the result should be interpreted cautiously.

These differences are descriptive and may reflect differences in loan programs, borrower profiles, pricing structures, and other factors rather than the effect of loan type alone.

## 7. Denial Reasons

This section examines the most commonly reported reasons for denied mortgage applications.

Because HMDA records may contain multiple denial reasons for a single application, denial-reason counts represent reported reasons rather than mutually exclusive application outcomes.

In [52]:
denied_df = decisioned_df[
    ~decisioned_df["approved"]
].copy()

denied_df.shape

(180539, 40)

In [53]:
denial_reason_label_cols = [
    "denial_reason_1_label",
    "denial_reason_2_label",
    "denial_reason_3_label",
    "denial_reason_4_label"
]

denied_df[denial_reason_label_cols].notna().sum()

denial_reason_1_label    180539
denial_reason_2_label     34538
denial_reason_3_label      5930
denial_reason_4_label      1041
dtype: int64

### 7.1 Reshape Denial Reasons

Because a denied application may report multiple denial reasons across four separate columns, we reshape the denial-reason fields into a long format before calculating reason frequencies.

In [54]:
denial_reasons_long = (
    denied_df[denial_reason_label_cols]
    .melt(
        value_name="denial_reason"
    )
    .dropna(subset=["denial_reason"])
)

denial_reasons_long.head()

,variable,denial_reason
0,denial_reason_1_label,Other
1,denial_reason_1_label,Unverifiable Information
2,denial_reason_1_label,Credit History
3,denial_reason_1_label,Credit History
4,denial_reason_1_label,Collateral


In [55]:
denial_reasons_long.shape

(222048, 2)

### 7.2 Most Common Denial Reasons

We calculate the frequency of each reported denial reason across all denied applications. Because a single application may report multiple reasons, these counts represent reported reasons rather than unique denied applications.

In [56]:
denial_reason_summary = (
    denial_reasons_long["denial_reason"]
    .value_counts()
    .rename_axis("denial_reason")
    .reset_index(name="reported_count")
)

denial_reason_summary

,denial_reason,reported_count
0,Debt-to-Income Ratio,74679
1,Credit History,37426
2,Credit Application Incomplete,30811
3,Collateral,28897
4,Other,25902
5,Unverifiable Information,11829
6,Insufficient Cash,9168
7,Employment History,2396
8,Exempt,867
9,Mortgage Insurance Denied,73


In [57]:
total_reported_reasons = len(denial_reasons_long)
total_denied_applications = len(denied_df)

denial_reason_summary["share_of_reported_reasons_pct"] = (
    denial_reason_summary["reported_count"]
    / total_reported_reasons
    * 100
)

denial_reason_summary["share_of_denied_applications_pct"] = (
    denial_reason_summary["reported_count"]
    / total_denied_applications
    * 100
)

denial_reason_summary

,denial_reason,reported_count,share_of_reported_reasons_pct,share_of_denied_applications_pct
0,Debt-to-Income Ratio,74679,33.631917,41.364470
1,Credit History,37426,16.854914,20.730147
2,Credit Application Incomplete,30811,13.875829,17.066119
3,Collateral,28897,13.013853,16.005960
4,Other,25902,11.665045,14.347039
5,Unverifiable Information,11829,5.327227,6.552047
6,Insufficient Cash,9168,4.128837,5.078127
7,Employment History,2396,1.079046,1.327137
8,Exempt,867,0.390456,0.480229
9,Mortgage Insurance Denied,73,0.032876,0.040434


### 7.3 Validate Denial Reason Summary

We verify that the summarized denial-reason counts reconcile with the total number of reported denial reasons in the reshaped dataset.

In [58]:
assert (
    denial_reason_summary["reported_count"].sum()
    == len(denial_reasons_long)
)

assert len(denied_df) == denied_count

print(f"Denied applications: {len(denied_df):,}")
print(f"Total reported denial reasons: {len(denial_reasons_long):,}")
print(f"Summarized denial reasons: {denial_reason_summary['reported_count'].sum():,}")
print("Validation passed.")

Denied applications: 180,539
Total reported denial reasons: 222,048
Summarized denial reasons: 222,048
Validation passed.


### 7.4 Key Findings on Denial Reasons

Debt-to-income ratio was the most frequently reported denial reason, appearing in approximately 41.4% of denied applications. Credit history was the second most common reason, reported in approximately 20.7% of denied applications.

Credit application incompleteness and collateral were also frequently reported, appearing in approximately 17.1% and 16.0% of denied applications, respectively.

Because a single denied application may contain multiple reported denial reasons, these percentages are not mutually exclusive and should not be summed to 100%.

The prominence of debt-to-income ratio is consistent with the earlier descriptive analysis, where applications with DTI above 50% showed substantially lower approval rates. However, these results remain descriptive and do not establish that DTI alone caused the observed lending outcomes.

## 8. Geographic Patterns

This section examines whether mortgage application volume and approval outcomes vary across California counties.

In [59]:
decisioned_df["county_code"].value_counts(dropna=False).head(15)

county_code
6037.0    141300
6065.0     68977
6073.0     66810
6059.0     54283
6071.0     50261
6067.0     35975
6085.0     27949
6001.0     26073
6013.0     24792
6029.0     19738
6019.0     19343
6077.0     18651
6111.0     15489
6061.0     13243
6095.0     10802
Name: count, dtype: int64

In [60]:
decisioned_df["county_code"].nunique(dropna=True)

58

### 8.1 County-Level Application Volume

We first examine the counties with the largest number of decisioned mortgage applications.

In [61]:
county_name_map = {
    6037: "Los Angeles",
    6065: "Riverside",
    6073: "San Diego",
    6059: "Orange",
    6071: "San Bernardino",
    6067: "Sacramento",
    6085: "Santa Clara",
    6001: "Alameda",
    6013: "Contra Costa",
    6029: "Kern",
    6019: "Fresno",
    6077: "San Joaquin",
    6111: "Ventura",
    6061: "Placer",
    6095: "Solano"
}

In [62]:
decisioned_df["county_name"] = (
    decisioned_df["county_code"]
    .map(county_name_map)
)

In [63]:
decisioned_df["county_name"].value_counts(dropna=False).head(15)

county_name
Los Angeles       141300
NaN               134225
Riverside          68977
San Diego          66810
Orange             54283
San Bernardino     50261
Sacramento         35975
Santa Clara        27949
Alameda            26073
Contra Costa       24792
Kern               19738
Fresno             19343
San Joaquin        18651
Ventura            15489
Placer             13243
Name: count, dtype: int64

### 8.2 Approval Rate by Major County

We compare approval outcomes across the 15 California counties with the largest decisioned application volumes. Counties outside this group remain in the dataset but are excluded from this focused comparison.

In [64]:
county_summary = (
    decisioned_df
    .dropna(subset=["county_name"])
    .groupby("county_name")
    .agg(
        applications=("approved", "count"),
        approved=("approved", "sum"),
        approval_rate=("approved", "mean")
    )
)

county_summary["denied"] = (
    county_summary["applications"]
    - county_summary["approved"]
)

county_summary["approval_rate_pct"] = (
    county_summary["approval_rate"] * 100
)

county_summary = county_summary.sort_values(
    "approval_rate_pct",
    ascending=False
)

county_summary

,applications,approved,approval_rate,denied,approval_rate_pct
county_name,,,,,
Santa Clara,27949,22459,0.803571,5490,80.357079
Placer,13243,10586,0.799366,2657,79.936570
Alameda,26073,20502,0.786331,5571,78.633069
Sacramento,35975,27742,0.771147,8233,77.114663
Orange,54283,41831,0.770610,12452,77.060958
Contra Costa,24792,18984,0.765731,5808,76.573088
Ventura,15489,11741,0.758022,3748,75.802182
Solano,10802,8174,0.756712,2628,75.671172
San Diego,66810,50298,0.752851,16512,75.285137


### 8.3 Validate Major-County Coverage

We verify that applications in the 15 mapped major counties and applications in the remaining California counties together reconcile with the full decisioned application sample.

In [65]:
major_county_count = decisioned_df["county_name"].notna().sum()
other_county_count = decisioned_df["county_name"].isna().sum()

assert major_county_count + other_county_count == len(decisioned_df)

assert (
    county_summary["applications"].sum()
    == major_county_count
)

print(f"Applications in 15 major counties: {major_county_count:,}")
print(f"Applications in remaining counties: {other_county_count:,}")
print(f"Total decisioned applications: {len(decisioned_df):,}")
print("Validation passed.")

Applications in 15 major counties: 593,686
Applications in remaining counties: 134,225
Total decisioned applications: 727,911
Validation passed.


### 8.4 Key Findings by County

Observed approval rates varied across major California counties.

Among the 15 counties with the largest decisioned application volumes, Santa Clara had the highest observed approval rate at approximately 80.4%, followed by Placer at 79.9% and Alameda at 78.6%.

At the lower end of the group, Fresno had an approval rate of approximately 71.9%, followed by San Bernardino at 72.6% and Los Angeles at 72.9%.

Other large counties also showed meaningful differences, including Orange County at approximately 77.1%, San Diego at 75.3%, and Riverside at 73.8%.

These geographic differences are descriptive and may reflect variation in borrower characteristics, loan products, local housing markets, and other factors across counties rather than the effect of location alone.

## 9. Overall Findings and Analytical Limitations

This section summarizes the main findings from the exploratory analysis and highlights important limitations that should be considered when interpreting the results.

### 9.1 Overall Findings

- The overall approval rate among 727,911 decisioned applications was approximately 75.2%.

- Approval outcomes varied substantially by loan purpose. Among the major categories, home purchase applications had the highest observed approval rate at approximately 88.4%, while other-purpose applications had the lowest at approximately 55.5%.

- Approval rates also differed across loan types. VA applications had the highest observed approval rate among the major loan types at approximately 82.8%, followed by FHA loans at 78.9% and conventional loans at 74.3%.

- Borrower financial characteristics showed meaningful associations with lending outcomes. Applications with DTI above 50% experienced sharply lower approval rates, while approval rates generally increased across higher reported income groups.

- LTV was not associated with approval outcomes in a simple linear pattern. Approval rates generally increased across several LTV ranges before declining modestly for applications with LTV of 100% or higher.

- Mortgage pricing varied across loan products and purposes. Conventional loans had higher median reported interest rates than FHA and VA loans, while home improvement and other-purpose loans had higher median rates than home purchase and refinancing loans.

- Debt-to-income ratio was the most frequently reported denial reason, appearing in approximately 41.4% of denied applications.

- Approval rates also varied geographically across major California counties, with observed rates ranging from approximately 71.9% to 80.4% among the 15 counties with the largest decisioned application volumes.

### 9.2 Analytical Limitations

The results in this notebook are descriptive and should not be interpreted as causal relationships.

Approval outcomes may be influenced by multiple factors simultaneously, including borrower income, debt-to-income ratio, loan amount, loan type, loan purpose, collateral characteristics, lender underwriting standards, and local housing-market conditions.

Several variables contain missing or incomplete information. For example, interest rates were unavailable for denied applications, which prevents direct pricing comparisons between approved and denied applications.

Extreme observations were preserved during data cleaning and excluded only from specific analyses when clearly flagged as potentially distortive. These exclusions were documented and reconciled with the full analysis sample.

Some categories, such as USDA/RHS loans and `Not Applicable` loan purposes, contained substantially fewer observations than the major categories and should therefore be interpreted cautiously.

County-level analysis focused on the 15 counties with the largest decisioned application volumes rather than all 58 California counties.

Finally, this exploratory analysis does not control for confounding factors. Multivariate modeling would be required to estimate the independent relationship between individual borrower or loan characteristics and approval outcomes.